In [ ]:

!pip install transformers accelerate bitsandbytes sentence-transformers huggingface-hub gradio chromadb

import torch
import gradio as gr
import json
import time
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
import chromadb
from google.colab import drive
import os


drive.mount('/content/drive')
save_path = '/content/drive/MyDrive/Bankacilik/'

print("="*60)
print("🏦  BANKA - AKILLI BANKACILIK ASİSTANI")
print("="*60)
print("📊 BERT Fine-Tune | RAG Bilgi Bankası | %96 Doğruluk")
print("="*60)


print("\n Modeller yükleniyor...")

bert_model_path = os.path.join(save_path, 'bert_intent_model_final')
classifier = pipeline("text-classification", model=bert_model_path, tokenizer=bert_model_path)

with open(os.path.join(save_path, 'kategori_haritasi_final.json'), 'r', encoding='utf-8') as f:
    kategori_haritasi = json.load(f)
id_to_kategori = kategori_haritasi['id_to_kategori']

embedder = SentenceTransformer('dbmdz/bert-base-turkish-128k-cased')
client = chromadb.PersistentClient(path=os.path.join(save_path, 'chroma_db_persistent'))
collection_name = "banking_rag_fast"

try:
    collection = client.get_collection(collection_name)
    print(f"✅ Mevcut vektör DB yüklendi")
except:
    rag_path = os.path.join(save_path, 'banka_bilgi_bankasi_rag_detayli.json')
    with open(rag_path, 'r', encoding='utf-8') as f:
        rag_bankasi = json.load(f)
    collection = client.create_collection(collection_name)
    chunk_id = 0
    for kategori, chunks in rag_bankasi.items():
        for chunk in chunks:
            embedding = embedder.encode(chunk).tolist()
            collection.add(
                embeddings=[embedding],
                documents=[chunk],
                metadatas=[{"kategori": kategori}],
                ids=[f"chunk_{chunk_id}"]
            )
            chunk_id += 1
    print(f"✅ {chunk_id} chunk indexlendi")



rag_cache = {}
bert_cache = {}

def rag_bul(mesaj):
    if mesaj in rag_cache:
        return rag_cache[mesaj]

    if mesaj in bert_cache:
        result = bert_cache[mesaj]
    else:
        result = classifier(mesaj)[0]
        bert_cache[mesaj] = result

    label_id = int(result['label'].split('_')[-1])
    intent = id_to_kategori[str(label_id)]
    confidence = float(result['score'])

    if confidence < 0.50:
        return None, intent, confidence, "dusuk_guven"

    soru_embedding = embedder.encode(mesaj).tolist()

    where_filter = {"kategori": intent}
    results = collection.query(
        query_embeddings=[soru_embedding],
        n_results=1,
        where=where_filter
    )

    if results['documents'] and len(results['documents'][0]) > 0:
        bilgi = results['documents'][0][0]
        rag_cache[mesaj] = (bilgi, intent, confidence, "rag_bulundu")
        return bilgi, intent, confidence, "rag_bulundu"

    results = collection.query(query_embeddings=[soru_embedding], n_results=1)
    if results['documents'] and len(results['documents'][0]) > 0:
        bilgi = results['documents'][0][0]
        rag_cache[mesaj] = (bilgi, intent, confidence, "rag_bulundu")
        return bilgi, intent, confidence, "rag_bulundu"

    rag_cache[mesaj] = (None, intent, confidence, "bulunamadi")
    return None, intent, confidence, "bulunamadi"

def cevap_uret(mesaj):
    basla = time.time()
    bilgi, intent, confidence, kaynak = rag_bul(mesaj)

    if not bilgi:
        sure = time.time() - basla
        return f"📌 Bu konuda bilgim yok. Lütfen bankacılık işlemlerinizle ilgili sorunuzu detaylandırın.\n\n---\n⏱️ {sure:.1f} saniye | ❌ {intent} (güven: {confidence:.0%})"

    if bilgi.startswith("📚") or bilgi.startswith("🏦"):
        cevap = bilgi
    else:
        cevap = f"📚 **{intent}**\n\n{bilgi}"

    sure = time.time() - basla
    cevap += f"\n\n---\n⏱️ {sure:.1f} saniye | 📚 {intent} (güven: {confidence:.0%})"
    return cevap

# GRADIO ARAYÜZÜ

with gr.Blocks(title="Mobil Bankacılık Danışmanı - Akıllı Bankacılık Asistanı", theme=gr.themes.Soft()) as demo:

    gr.HTML(f"""
    <div style='text-align: center; background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%); padding: 20px; border-radius: 15px; margin-bottom: 20px;'>
        <h1 style='color: white; font-size: 36px; margin: 0;'>🏦 Banka </h1>
        <h3 style='color: #e0e0ff; margin: 5px 0;'>Akıllı Bankacılık Asistanı</h3>
        <p style='color: #b0b0ff; margin: 0;'>BERT Fine-Tune | RAG Bilgi Bankası | %96 Doğruluk | 16 Kategori</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=550, bubble_full_width=False, avatar_images=(None, "🏦"))

            with gr.Row():
                msg = gr.Textbox(
                    label="Mesajınız",
                    placeholder="Örn: Kredi kartımı nasıl aktif ederim?",
                    lines=2,
                    scale=4
                )
                send = gr.Button("📤 Gönder", variant="primary", scale=1)

            with gr.Row():
                clear = gr.Button("🗑️ Temizle", variant="secondary")
                btn1 = gr.Button("🛑 Kart Kaybı", variant="secondary")
                btn2 = gr.Button("🏠 Konut Kredisi", variant="secondary")
                btn3 = gr.Button("💳 Kart Aktivasyon", variant="secondary")

        with gr.Column(scale=1):
            gr.HTML("""
            <div style='background: #f0f4ff; padding: 15px; border-radius: 10px;'>
                <h4>⚡ Sistem Özellikleri</h4>
                <p>✅ BERT Fine-Tune - %96 doğruluk</p>
                <p>✅ RAG Bilgi Bankası - 50+ bilgi chunk'ı</p>
                <p>✅ 16 Kategori - Kapsamlı destek</p>
                <p>✅ 2-3 Saniye yanıt süresi</p>
            </div>

            <div style='background: #fff3cd; padding: 15px; border-radius: 10px; margin-top: 15px;'>
                <h4>🎯 Test Soruları</h4>
                <p>• Kartım kayboldu ne yapmalıyım?</p>
                <p>• Konut kredisi faiz oranları nedir?</p>
                <p>• Kredi kartı nasıl aktif edilir?</p>
                <p>• İhtiyaç kredisi için gerekli belgeler</p>
            </div>

            <div style='background: #e8f5e8; padding: 15px; border-radius: 10px; margin-top: 15px;'>
                <h4>📞 Acil Durum</h4>
                <p><b>Kayıp/Çalıntı Kart:</b> 0850 123 45 67</p>
                <p><b>Müşteri Hizmetleri:</b> 0850 123 45 68</p>
            </div>
            """)

    def respond(message, history):
        if not message.strip():
            return "", history
        cevap = cevap_uret(message)
        history.append((message, cevap))
        return "", history

    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    send.click(respond, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: ([], ""), None, [chatbot, msg])
    btn1.click(lambda: "Kartım kayboldu ne yapmalıyım?", None, msg)
    btn2.click(lambda: "Konut kredisi faiz oranları nedir?", None, msg)
    btn3.click(lambda: "Kredi kartı nasıl aktif edilir?", None, msg)





demo.launch(share=True, server_port=7860)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Fou

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/386 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/740M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

✅ Mevcut vektör DB yüklendi


/tmp/ipykernel_5940/4108800725.py:122: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Mobil Bankacılık Danışmanı - Akıllı Bankacılık Asistanı", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_5940/4108800725.py:134: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=550, bubble_full_width=False, avatar_images=(None, "🏦"))
/tmp/ipykernel_5940/4108800725.py:134: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(height=550, bubble_full_width=False, avatar_images=(None, "🏦"))
/tmp/i

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://782938ed4ceee3583b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
